In [0]:
dbutils.widgets.dropdown("data_source", "order_items", ["order_items"], "Data Source")
dbutils.widgets.dropdown("catalog", "dev", ["dev", "prod"])

data_source = dbutils.widgets.get("data_source")
catalog = dbutils.widgets.get("catalog")

print(f"Selected source: {data_source} and catalog: {catalog}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, DateType, TimestampType, DoubleType, LongType

from pyspark.sql.window import Window

from delta.tables import DeltaTable

In [0]:
silver_order_items_schema = StructType([
    StructField("order_item_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("sku", StringType(), True),
    StructField("quantity", LongType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("line_total", DoubleType(), True),
    StructField("return_requested", BooleanType(), True),
    StructField("return_reason", StringType(), True),
    StructField("_ingested_at", TimestampType(), True),
    StructField("_source_file", StringType(), True),
])

In [0]:
source_table = "dev.os_stepright.bronze_order_items_valid"
target_table = "dev.os_stepright.silver_order_items_all"
checkpoint_loc = f"/Volumes/{catalog}/os_stepright/checkpoints/silver_{data_source}_all"

In [0]:
def target_table_existense_check(target_table, schema):

    if not spark.catalog.tableExists(target_table):

        empty_df = spark.createDataFrame([], schema)
        empty_df.write.format("delta").saveAsTable(target_table)

    else:
        pass

In [0]:
# ORDER_ITEMS_RULES = {
#     "return_has_reason": "NOT (return_requested = true AND return_reason IS NULL)",
#     "line_total_matches": "(ABS(line_total - (quantity * unit_price)) < 0.01)",
# }

# expression = " OR ".join([f"{i}" for i in ORDER_ITEMS_RULES.values()])

In [0]:
def read_source(source_table):

    source_df = (spark.readStream.table(source_table)
             .selectExpr("after.*", "ts_ms","op", "_ingested_at", "_source_file")
             .withColumn("ts_ms", (F.col("ts_ms") / 1000).cast("timestamp"))
    )

    return source_df

In [0]:
target_table_existense_check(target_table, silver_order_items_schema)

In [0]:
def process_batch(batchDf, batchId):

    window_spec = Window.partitionBy(F.col("order_item_id")).orderBy(F.col("ts_ms").desc())

    dedup_df = (batchDf.withColumn("rn", F.row_number().over(window_spec))
                    .filter("rn=1")
                    .drop('rn')
    )

    target_delta = DeltaTable.forName(spark, target_table)

    (target_delta.alias("t")
     .merge(dedup_df.alias("s"), "t.order_item_id==s.order_item_id")
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute()
    )

source_df_stream = read_source(source_table)

(source_df_stream.writeStream
 .foreachBatch(process_batch)
 .option("checkpointLocation", checkpoint_loc)
 .trigger(availableNow=True)
 .start()
)

In [0]:
ORDER_ITEMS_RULES = {
    "return_has_reason": "(NOT (return_requested = true AND return_reason IS NULL))",
    "line_total_matches": "(ABS(line_total - (quantity * unit_price)) < 0.01)",
}

quarantine_expression = " AND ".join([f"{i}" for i in ORDER_ITEMS_RULES.values()])

In [0]:
print(quarantine_expression)

In [0]:
second_source= "dev.os_stepright.silver_order_items_all"
order_items_clean_target_table = f"{catalog}.os_stepright.silver_order_items"
order_items_quarantined_target_table = f"{catalog}.os_stepright.silver_order_items_quarantined"
quarantine_split_checkpoint_loc = f"/Volumes/{catalog}/os_stepright/checkpoints/silver_{data_source}_quarantined_split"

In [0]:
silver_order_items_schema = StructType([
    StructField("order_item_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("sku", StringType(), True),
    StructField("quantity", LongType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("line_total", DoubleType(), True),
    StructField("return_requested", BooleanType(), True),
    StructField("return_reason", StringType(), True),
    StructField("_ingested_at", TimestampType(), True),
    StructField("_source_file", StringType(), True),
])

In [0]:
target_table_existense_check(order_items_clean_target_table, silver_order_items_schema)
target_table_existense_check(order_items_quarantined_target_table, silver_order_items_schema)



In [0]:
def process_quarantine_split_batch(batchdf, batchid):

    full_df = (spark.table(second_source)
        .withColumn("is_quarantined", ~F.expr(quarantine_expression))
    )

    clean_df = full_df.filter("is_quarantined = false").drop("is_quarantined")
    quarantined_df = full_df.filter("is_quarantined = true").drop("is_quarantined")

    clean_df.write.format("delta").mode("overwrite").saveAsTable(order_items_clean_target_table)
    quarantined_df.write.format("delta").mode("overwrite").saveAsTable(order_items_quarantined_target_table)

In [0]:
(spark.readStream.table(second_source)
    .writeStream
    .foreachBatch(process_quarantine_split_batch)
    .option("checkpointLocation", quarantine_split_checkpoint_loc)
    .trigger(availableNow=True)
    .start()
)